In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import tacco as tc
import os

In [ ]:
## load clinical informaiton
clinical_data = pd.read_csv('../data/clinical.csv')

In [ ]:
clinical_data['batch'] = clinical_data['Sample_ID'] + '_' + clinical_data['Core']
clinical_data

In [ ]:
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt


## Data preprocessing
non_sus = [
    "P-1S-R-3K-B05-54-D",       # Exclude the entire sample
    "P-1S-R-3K-B13-54-D_C00",   # Exclude only this specific core
]

cutoff_list = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


# ============================================================
# 1. Load each H5AD file once and split it by core
# ============================================================

adata_list = []

for sample in clinical_data["Sample_ID"].dropna().unique():

    # Exclude an entire sample when its original sample ID is in non_sus
    if sample in non_sus:
        continue

    adata = sc.read_h5ad(
        f"../data/h5ad/{sample}.h5ad"
    )

    core_names = adata.obs["core_name"].dropna().unique()
    n_cores = len(core_names)

    for core in core_names:

        core_mask = adata.obs["core_name"] == core
        adata_core = adata[core_mask].copy()

        # If the sample contains only one core, use the original sample name.
        # Otherwise, append the core name.
        if n_cores == 1:
            sample_name = sample
        else:
            sample_name = f"{sample}_{core}"

        # Exclude a specific sample-core combination
        if sample_name in non_sus:
            continue

        # Store sample information in AnnData
        adata_core.obs["sample_name"] = sample_name

        adata_list.append(adata_core)


# ============================================================
# 2. Calculate ratio_counts for each cutoff
# ============================================================

qc_list = []

for adata_core in adata_list:

    sample_name = adata_core.obs["sample_name"].iloc[0]
    total_counts = adata_core.obs["total_counts"]

    for cutoff in cutoff_list:

        ratio_counts = (total_counts > cutoff).mean()

        qc_list.append(
            {
                "sample_name": sample_name,
                "cutoff": cutoff,
                "ratio_counts": ratio_counts,
            }
        )


qc_df = pd.DataFrame(qc_list)

# Optional sorting
qc_df = qc_df.sort_values(
    ["sample_name", "cutoff"]
).reset_index(drop=True)

qc_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sample_name, sample_df in qc_df.groupby("sample_name"):

    sample_df = sample_df.sort_values("cutoff")

    ax.plot(
        sample_df["cutoff"],
        sample_df["ratio_counts"],
        marker="o",
        linewidth=1.5,
        markersize=4,
        alpha=0.8,
        label=sample_name,
    )

ax.set_xlabel("Total count cutoff")
ax.set_ylabel("Proportion of bins passing cutoff")
ax.set_title("Changes in QC-passing bin ratios by total-count cutoff")

ax.set_xticks(cutoff_list)
ax.set_ylim(0, 1)

ax.grid(
    axis="both",
    linestyle="--",
    alpha=0.3,
)

ax.legend(
    title="Sample",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

plt.tight_layout()

# Save before plt.show()
fig.savefig(
    "../figures/ratio_counts_by_cutoff.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()